# P11-T06 — Fresh Colab T4 Acceptance

Notebook điều phối cho Chương. Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt hoặc **Run all**. Upload ZIP source local và JSON sidecar khi hộp thoại xuất hiện; dataset chính thức được tải từ Drive bằng loader của nhóm.

Đây là workflow **candidate** cho source tích hợp có thay đổi sau freeze T02. Phạm vi src/build tại pin 696fc82 đã được đối chiếu với freeze f2837a8; ZIP HEAD mới vẫn cần nghiệm thu riêng. Nó không sửa notebook tác giả, không push GitHub và không tự chứng nhận số liệu hoặc PDF. Các vấn đề N01–N07 và nội dung T04 phải được xử lý trước nghiệm thu cuối. Hướng dẫn đầy đủ nằm trong `docs/report/p11-t06-colab-runbook.md` của ZIP.


In [ ]:
# Use archive mode while the acceptance branch remains unpublished.
SOURCE_MODE = "archive"  # "archive" or "published"
FROZEN_SOURCE_SHA = "696fc82c72b2098ea7ca24c7c89efdab9d3452ec"  # Exact public 40-character SHA supplied by T02, for published mode.
FREEZE_CONFIRMED = False  # Set only after T02 confirms this exact source revision.
DATASET_DIR = "/content/group16-dataset"
DOWNLOAD_INDIVIDUAL_FILES = False


## 1. Install Acceptance Dependencies

Cell này cài công cụ notebook/PDF, không thay CUDA toolkit. Log setup được giữ để đưa vào gói evidence. Chromium tải MathJax khi xuất HTML; nếu mạng/CDN lỗi, exporter giữ log và không coi TeX thô là công thức đã render.


In [ ]:
import subprocess, sys, tempfile, json, hashlib, zipfile, stat, shutil
from pathlib import Path, PurePosixPath

SETUP_ROOT = Path(tempfile.mkdtemp(prefix="group16-p11-bootstrap-", dir="/content"))
with (SETUP_ROOT / "setup.log").open("w", encoding="utf-8") as setup_log:
    for command in (
        [sys.executable, "-m", "pip", "install", "-q", "nbformat", "nbconvert", "nbclient", "ipykernel",
         "playwright", "pypdf", "pillow", "numpy", "matplotlib", "gdown"],
        [sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"],
    ):
        setup_log.write(repr(command) + "\n")
        setup_log.flush()
        result = subprocess.run(command, stdout=setup_log, stderr=subprocess.STDOUT)
        if result.returncode:
            print((SETUP_ROOT / "setup.log").read_text(encoding="utf-8")[-6000:])
            raise RuntimeError(f"Acceptance dependencies failed; retained log: {SETUP_ROOT / 'setup.log'}")
print("Setup log:", SETUP_ROOT / "setup.log")


## 2. Upload and Verify the Local Bundle

Upload **hai file** từ local: `Group16_P11_T06_Source.zip` và `Group16_P11_T06_Source.json`. Có thể upload thêm `Group16_Final.ipynb` mới nhất của Thành nếu đã chốt T04; input này được lưu riêng và có hash. Nếu không có notebook rời, dùng bản đã commit trong ZIP.

Cả hai source mode đều dùng ZIP local để mang công cụ acceptance chưa công bố lên Colab. Mode `published` chỉ lấy production source từ SHA public chỉ định; không lấy production source từ nhánh local hoặc fallback `main`.


In [ ]:
from google.colab import files

uploaded = files.upload()
archives = [Path(name).resolve() for name in uploaded if name.endswith(".zip")]
sidecars = [Path(name).resolve() for name in uploaded if name.endswith(".json")]
if len(archives) != 1 or len(sidecars) != 1:
    raise ValueError("Upload exactly one source ZIP and its JSON sidecar.")
SOURCE_ARCHIVE = archives[0]
SOURCE_META = json.loads(sidecars[0].read_text(encoding="utf-8"))
actual_hash = hashlib.sha256(SOURCE_ARCHIVE.read_bytes()).hexdigest()
if actual_hash != SOURCE_META["archive_sha256"]:
    raise ValueError("Uploaded source archive checksum differs from local sidecar.")
TOOLS_ROOT = Path(tempfile.mkdtemp(prefix="group16-p11-tools-", dir="/content"))
with zipfile.ZipFile(SOURCE_ARCHIVE) as bundle:
    entries = bundle.infolist()
    if sum(entry.file_size for entry in entries) > 256 * 1024 * 1024:
        raise ValueError("Source ZIP exceeds the uncompressed size budget.")
    seen = set()
    for entry in entries:
        name = entry.filename.replace("\\", "/")
        target = (TOOLS_ROOT / name).resolve()
        if (not target.is_relative_to(TOOLS_ROOT) or target == TOOLS_ROOT or target in seen
                or stat.S_ISLNK(entry.external_attr >> 16) or ":" in name):
            raise ValueError(f"Unsafe or duplicate source entry: {name}")
        seen.add(target)
    for entry in entries:
        target = TOOLS_ROOT / entry.filename.replace("\\", "/")
        if entry.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(bundle.read(entry))
sys.path.insert(0, str(TOOLS_ROOT / "scripts"))
from package_notebook_acceptance import verify_tree, full_sha
verify_tree(TOOLS_ROOT, SOURCE_META["source_sha"])
SOURCE_SHA = (full_sha(FROZEN_SOURCE_SHA) if SOURCE_MODE == "published"
              else SOURCE_META["source_sha"])
REPORT_NOTEBOOK = next((Path(n).resolve() for n in uploaded if Path(n).name == "Group16_Final.ipynb"), None)
print("Tools SHA:", SOURCE_META["source_sha"])
print("Effective production source SHA:", SOURCE_SHA)
print("Source mode:", SOURCE_MODE, "| source freeze declared:", FREEZE_CONFIRMED)


## 3. Execute the Report and Preserve Evidence

Controller kiểm T4/sm_75, driver, nvcc, bộ nhớ và dataset; build bằng `scripts/build.py --compiler nvcc --require-t4` (sm_75, tự bật V5 khi có API). Bản chạy thay bootstrap và truyền danh sách V0–V5 rõ ràng cho producer; toàn bộ Markdown và các cell thuật toán/bảng/biểu đồ giữ nguyên. Bản input, runtime overlay và hash đều được lưu.

Không đưa thời gian Python điều phối vào benchmark. Thời gian trong bảng vẫn do binary `stitch` đo. Lỗi dừng notebook báo cáo và giữ partial output/log/ZIP; cell download phía sau vẫn cho tải evidence để kiểm tra.


In [ ]:
EVIDENCE_ROOT = Path("/content/results/p11-t06")
command = [sys.executable, str(TOOLS_ROOT / "scripts/accept_final_notebook.py"),
           "--source-mode", SOURCE_MODE, "--source-sha", SOURCE_SHA,
           "--dataset", DATASET_DIR, "--evidence-root", str(EVIDENCE_ROOT)]
if SOURCE_MODE == "archive":
    command += ["--source-archive", str(SOURCE_ARCHIVE), "--archive-sha256", actual_hash]
if FREEZE_CONFIRMED:
    command += ["--frozen"]
if REPORT_NOTEBOOK:
    command += ["--notebook", str(REPORT_NOTEBOOK)]
print("Running:", " ".join(command))
with (SETUP_ROOT / "controller.log").open("w", encoding="utf-8") as controller_log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        controller_log.write(line)
        controller_log.flush()
        print(line, end="", flush=True)
    return_code = process.wait()
print("Controller exit code:", return_code)
runs = [p for p in EVIDENCE_ROOT.iterdir() if (p / "execution_summary.json").is_file()]
RUN = max(runs, key=lambda p: p.stat().st_mtime)
summary = json.loads((RUN / "execution_summary.json").read_text(encoding="utf-8"))
print("Run status:", summary["status"])
print("Final acceptance remains pending manual numerical/gallery/PDF review and T02/T04.")


## 4. Validate and Download Artifacts

ZIP chứa notebook có output (hoặc partial output nếu lỗi), manifest source/GPU/dataset, logs, raw benchmark, gallery, kiểm số liệu và PDF nếu xuất được. Bản PDF dùng tên `Group16_Final.pdf` trong thư mục `pdf/`. Cell này không mount hoặc ghi Drive; có thể tự lưu ZIP lên Drive sau khi tải.

Dù kiểm tự động đạt, vẫn cần kiểm số trong lời văn, cặp mẫu môn học và render từng trang PDF. `executed_review_pending` không phải chứng nhận hoàn tất T06.


In [ ]:
from accept_final_notebook import validate_artifacts, bundle

shutil.copytree(SETUP_ROOT, RUN / "bootstrap_logs", dirs_exist_ok=True)
if summary["status"] == "executed_review_pending":
    print(json.dumps(validate_artifacts(RUN), ensure_ascii=False, indent=2))
else:
    print("Execution failed; download the retained evidence before retrying in a new runtime.")
archive = bundle(RUN)
print("Evidence archive:", archive)
files.download(str(archive))
if DOWNLOAD_INDIVIDUAL_FILES:
    for artifact in (RUN / "Group16_Final.ipynb", RUN / "pdf/Group16_Final.pdf"):
        if artifact.is_file():
            files.download(str(artifact))
